# Filter the processed datasets

In [ ]:
# Imports:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.impute import KNNImputer, SimpleImputer

from tableone import TableOne

from dataset_features import *
from experiment_config import *

In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)

## 1. Select table

In [ ]:
dataset = "mimiciv"
data_path = f"./processed_data/{dataset}_hfno_data.csv"

orig_df = pd.read_csv(data_path)
orig_df

In [ ]:
print(f"Step 1 - Loaded dataset: {len(orig_df)} patients (after DNI/CMO exclusion in notebook 1)")
orig_df

#### Define HFNO failure outcome variable

In [ ]:
orig_df['hfno_failure'] = np.where(
    (orig_df['intubated'] == 1) | (orig_df['died'] == 1), 
    1,
    0
)

#### Split dataset into separate tables

In [ ]:
LASTDATA_FEATURES = {
    "all": dataset_features["base_features"] + dataset_features["comorbidities"] + dataset_features["vitals_mean_last24h"] + dataset_features["last"] + ['fio2_last', 'flow_rate_last', 'resp_rate_last', 'spo2_last'],
    "base_features": dataset_features["base_features"],
    "comorbidities": dataset_features["comorbidities"],
    "vitals": dataset_features["vitals_mean_last24h"],
    "bg_and_lab": dataset_features["last"],
    "settings": dataset_features['settings']
}


features = LASTDATA_FEATURES
df = orig_df[features["all"]]
df

In [ ]:
time_df = orig_df[dataset_features['base_features'] + dataset_features['timestamp_features']]
time_df

In [ ]:
# sepsis_df = orig_df[orig_df.sepsis3 == 1]
# sepsis_df = sepsis_df[['stay_id'] + dataset_features['sepsis3_components']]
sepsis_df = orig_df[dataset_features['base_features'] + dataset_features['sepsis3_components']]
sepsis_df

## 2. Remove patients with >30% missing data

In [ ]:
# patient_missing_df = (df.isnull().sum(axis=1) / len(df.columns)).sort_values(ascending=False)
temp = df.set_index('stay_id')
# temp = df.set_index('stay_id').drop(columns=['flow_rate_last', 'resp_rate_last', 'fio2_last'])
patient_missing_df = (temp.isnull().sum(axis=1) / len(temp.columns)).sort_values(ascending=False)
patient_missing_df

In [ ]:
excluded_patients = patient_missing_df[patient_missing_df > 0.30]
excluded_patient_ids = excluded_patients.index
print(f"Step 2 - Excluded patients (>30% missing data): {len(excluded_patient_ids)}")
print(f"Step 2 - Remaining after patient exclusion: {len(patient_missing_df) - len(excluded_patient_ids)} patients")
excluded_patients

In [ ]:
df = df[~df['stay_id'].isin(excluded_patient_ids)]
df
print(f"  -> Remaining patients in df after exclusion: {len(df)}")


In [ ]:
time_df = time_df[~time_df['stay_id'].isin(excluded_patient_ids)]
time_df

In [ ]:
sepsis_df = sepsis_df[~sepsis_df['stay_id'].isin(excluded_patient_ids)]
sepsis_df

## 3. Remove features with >25% missing values

In [ ]:
vars = df.columns
feature_missing_df = (df[vars].isna().sum() / len(df.index)).sort_values(ascending=False)
feature_missing_df

In [ ]:
excluded_features = feature_missing_df[feature_missing_df > 0.25]
print(f"Step 3 - Excluded features (>25% missing data): {len(excluded_features.index)}")
print(f"Step 3 - Remaining features: {len(feature_missing_df) - len(excluded_features.index)}")
excluded_features_list = excluded_features.index.tolist()
excluded_features

In [ ]:
df_filtered = df.drop(excluded_features.index.tolist(), axis=1)
df_filtered

In [ ]:
# Optional: categorise ethnicities
race_white_labels = [
    'WHITE',
    'WHITE - EASTERN EUROPEAN',
    'WHITE - RUSSIAN',
    'WHITE - OTHER EUROPEAN',
    'WHITE - BRAZILIAN'
]
race_black_labels = [
    'BLACK/AFRICAN AMERICAN',
    'BLACK/CAPE VERDEAN',
    'BLACK/CARIBBEAN ISLAND',
    'BLACK/AFRICAN'
]
race_asian_labels = [
    'ASIAN',
    'ASIAN - SOUTH EAST ASIAN',
    'ASIAN - CHINESE',
    'ASIAN - ASIAN INDIAN'
]
race_hispanic_labels = [
    'HISPANIC/LATINO - SALVADORAN', 
    'HISPANIC/LATINO - GUATEMALAN',
    'HISPANIC/LATINO - PUERTO RICAN',
    'HISPANIC/LATINO - CENTRAL AMERICAN', 
    'HISPANIC/LATINO - DOMINICAN',
    'HISPANIC/LATINO - MEXICAN',
    'HISPANIC/LATINO - COLUMBIAN',
    'HISPANIC OR LATINO'
]
race_unknown_labels = [
    'UNKNOWN',
    'UNABLE TO OBTAIN',
    'PATIENT DECLINED TO ANSWER',
]
race_not_other_labels = race_white_labels + race_black_labels + race_asian_labels + race_hispanic_labels + race_unknown_labels

df_filtered['race_cat'] = df_filtered['race']

df_filtered.loc[df_filtered['race'].isin(race_white_labels), 'race_cat'] = 'WHITE'
df_filtered.loc[df_filtered['race'].isin(race_black_labels), 'race_cat'] = 'BLACK'
df_filtered.loc[df_filtered['race'].isin(race_asian_labels), 'race_cat'] = 'ASIAN'
df_filtered.loc[df_filtered['race'].isin(race_hispanic_labels), 'race_cat'] = 'HISPANIC'
df_filtered.loc[df_filtered['race'].isin(race_unknown_labels), 'race_cat'] = 'UNKNOWN'
df_filtered.loc[~df_filtered['race'].isin(race_not_other_labels), 'race_cat'] = 'OTHER'
df_filtered['race_cat'].unique()

## 4. Combine / Categorise features

In [ ]:
# Categorise GCS_min into 4 bins (Severe/Moderate/Mild/Normal):
gcs_bins   = [2, 8, 12, 14, 15]
gcs_labels = ["Severe (3-8)", "Moderate (9-12)", "Mild (13-14)", "Normal (15)"]
df_filtered['gcs_binned'] = pd.cut(df_filtered['gcs_min'], bins=gcs_bins, labels=gcs_labels)

# Convert gender into binary labels:
df_filtered['gender'] = (df_filtered['gender'] == 'M').astype(int)

df_filtered[['gcs_min', 'gcs_binned', 'gender']]

## 5. Examine feature distributions

### 5.1 Demographics & Disease severity

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16,12))

df_filtered['admission_age'].hist(ax=axes[0, 0]).set_title('Age (years)')
df_filtered['gender'].hist(ax=axes[0, 1]).set_title('Gender (M/F)')
df_filtered['weight_admit'].hist(ax=axes[1, 0]).set_title('Weight (kg)')
df_filtered['race_cat'].hist(ax=axes[1, 1]).set_title('Ethnicity')

# df_filtered['gcs_min'].hist(ax=axes[2, 0]).set_title('Min. GCS')
df_filtered['gcs_binned'].hist(ax=axes[2, 0]).set_title('Min. GCS')

df_filtered['apsiii'].hist(ax=axes[2, 1]).set_title('APSIII')


In [ ]:
df_filtered[['admission_age', 'weight_admit']].describe()

In [ ]:
df_filtered[['gender', 'race_cat']].describe()

### 5.2 Comorbidities

In [ ]:
comorbs = features["comorbidities"]
print("Fractions:")
print(df_filtered[comorbs].sum() / len(df_filtered.index))
print("\nCounts:")
print(df_filtered[comorbs].sum())
df_filtered[comorbs].describe()

### 5.3 Vital signs

In [ ]:
vitals = features["vitals"]

fig, axes = plt.subplots(3, 3, figsize=(16,12))

for i, var in enumerate(vitals):
    row = i // 3
    col = i % 3
    df_filtered[var].hist(ax=axes[row, col]).set_title(var)

### 5.4 Lab values and Blood gasses

In [ ]:
lab_bg_vars = list(set(features["bg_and_lab"]) - set(excluded_features_list)) + ['fio2_last', 'flow_rate_last', 'resp_rate_last', 'spo2_last']

fig, axes = plt.subplots(10, 3, figsize=(16,36))

for i, var in enumerate(lab_bg_vars):
    row = i // 3
    col = i % 3
    df_filtered[var].hist(ax=axes[row, col]).set_title(var)

In [ ]:
df_filtered['fluidbalance_24hr'] = df_filtered['fluidbalance_24hr'].clip(lower=-4000, upper=5000)
df_filtered['fluidbalance_24hr'].hist()


In [ ]:
df_filtered['vp_last6h'].describe()

### 5.5 Outcome variable: Intubation

In [ ]:
df_filtered['intubated'].value_counts()

### 5.6 Add ROX variable

In [ ]:
df_filtered['rox_last'] = (df_filtered['spo2_last'] / (df_filtered['fio2_last']/100)) / df_filtered['resp_rate_last']
df_filtered.drop(columns=['resp_rate_last'], inplace=True)
df_filtered['rox_last'].describe()

## 6. Examination using the tableone package

In [ ]:
df_filtered.columns

In [ ]:
df_filtered = df_filtered.drop(columns=['stay_id'])
if 'resp_rate_last' in df_filtered.columns:
    df_filtered = df_filtered.drop(columns=['resp_rate_last'])

# categorical = ['gender', 'race_cat', 'gcs_binned'] + features["comorbidities"]
categorical = ['inhosp_mortality', 'gender', 'race_cat', 'gcs_binned'] + features["comorbidities"] + ['vp_last6h']

numerical = ['admission_age','weight_admit','apsiii'] + features['vitals'] + features['bg_and_lab'] + ['spo2_last', 'fio2_last', 'flow_rate_last', 'rox_last']
numerical = list(set(numerical) - set(excluded_features_list) - set(NUMERICAL_CORRELATED))

columns = categorical + numerical

In [ ]:
# Tableone formatting parameters (limit / order / labels / decimals) come from
# experiment_config.py, imported above. Previously duplicated here, which meant
# edits to the config had no effect on this notebook.


In [ ]:
# for col in numerical:
#     print(f"\'{col}\': \'\',")

ordered_cols = list(FORMAT_PARAMS['labels'].keys())
print(columns)
print(ordered_cols)

print(set(ordered_cols) - set(columns))
print(set(columns) - set(ordered_cols))

print()
if set(columns) == set(ordered_cols):
    print("Replacing columns with ordered_cols.")
    columns = ordered_cols
else:
    print("There was a mismatch between columns and the specified ordered columns. Defaulting to given columns order.")

In [ ]:
# Print table 1 with information on HFNO duration and time until HFNO initiation:
df_filtered['time_to_hfno'] = pd.to_timedelta(pd.to_datetime(time_df.final_starttime) - pd.to_datetime(time_df.admittime)).dt.total_seconds().astype(int) / 60 / 60
df_filtered['time_on_hfno'] = pd.to_timedelta(pd.to_datetime(time_df.final_endtime) - pd.to_datetime(time_df.final_starttime)).dt.total_seconds().astype(int) / 60 / 60

table_hfnofailure_time = TableOne(
    data=df_filtered,
    columns=columns + ['time_to_hfno', 'time_on_hfno'],
    categorical=categorical,
    nonnormal=NON_NORMAL_VARS + ['spo2_last', 'time_to_hfno', 'time_on_hfno'],
    groupby='hfno_failure',
    limit=FORMAT_PARAMS['limit'],
    order=FORMAT_PARAMS['order'],
    labels=FORMAT_PARAMS['labels'],
    decimals=FORMAT_PARAMS['decimals'],
    sort=False,
    # htest_name=True,
    pval=True
)
table_hfnofailure_time

In [ ]:
df_tabledata = df_filtered[columns]

# table_hfnofailure = TableOne(
#     data=df_filtered,
#     columns=columns,
#     categorical=categorical,
#     nonnormal=NON_NORMAL_VARS,
#     groupby='hfno_failure',
#     dip_test=True,
#     normal_test=True,
#     tukey_test=True,
#     pval=True
# )

table_hfnofailure = TableOne(
    data=df_filtered,
    columns=columns,
    categorical=categorical,
    nonnormal=NON_NORMAL_VARS,
    groupby='hfno_failure',
    limit=FORMAT_PARAMS['limit'],
    order=FORMAT_PARAMS['order'],
    labels=FORMAT_PARAMS['labels'],
    decimals=FORMAT_PARAMS['decimals'],
    rename=FORMAT_PARAMS['labels'],
    sort=False,
    # htest_name=True,
    pval=True
)
table_hfnofailure.to_latex('results/HFNOstudy_Table1_HFNOfailure.tex')
table_hfnofailure.to_csv('results/HFNOstudy_Table1_HFNOfailure.csv')
table_hfnofailure

In [ ]:
table_intubated = TableOne(
    data=df_filtered,
    columns=columns,
    categorical=categorical,
    nonnormal=NON_NORMAL_VARS,
    groupby='intubated',
    dip_test=True,
    normal_test=True,
    tukey_test=True,
    pval=True
)
table_intubated

## 7. Examine correlation between features

In [ ]:
corr_matrix = df_tabledata.corr(numeric_only=True)
corr_matrix.style.apply(lambda x: ["background: red" if v > 0.6 else "" for v in x], axis = 1)

In [ ]:
large_corr_matrix = corr_matrix[corr_matrix[corr_matrix >= 0.6].count() > 1]
large_corr_matrix.style.apply(lambda x: ["background: red" if v > 0.6 else "" for v in x], axis = 1)

In [ ]:
"""
To exclude:
- SBP & DBP --> Keep MBP
- Hematocrit & RBC --> Keep Hemoglobin
- MCV --> Keep MCH
- Baseexcess --> Keep Bicarbonate
- BUN --> Keep creatinine
- Chloride --> Keep sodium
- TotalCO2 --> Keep PaCO2
- Last Glucose --> Keep mean glucose over last 24h
"""

## 8. Impute missing data

In [ ]:
# Split numerical and categorical data into separate dataframes:
df_table_reindex = df_tabledata.reset_index(drop=True)
df_table_reindex.to_csv(f"filtered_data/{dataset}_tabledata.csv")

df_numerical = df_table_reindex[numerical]
df_categorical = df_table_reindex[categorical]
df_numerical

### 8.1 Numerical data imputation

In [ ]:
# # Impute numerical data with K-Nearest Neighbour (KNN) imputation:
# imputer = KNNImputer(n_neighbors=5, weights="uniform")
# numerical_imp = imputer.fit_transform(df_numerical)
# df_numerical_imp = pd.DataFrame(data=numerical_imp, columns=df_numerical.columns)
# df_numerical_imp

# Impute numerical data with median values:
imputer = SimpleImputer(strategy='median')
numerical_imp = imputer.fit_transform(df_numerical)
df_numerical_imp = pd.DataFrame(data=numerical_imp, columns=df_numerical.columns)
df_numerical_imp


### 8.2 Categorical data imputation

In [ ]:
# Impute categorical data with most frequent value:
df_categorical_imp = df_categorical
categorical_missing = (df_categorical[df_categorical.columns].isna().sum()).sort_values(ascending=False)
cat_missing_vars = categorical_missing[categorical_missing > 0].index.tolist()
print(f"Categorical features with missing values: {cat_missing_vars}")
categorical_missing

In [ ]:
for var in cat_missing_vars:
    mode = df_categorical[var].mode()[0]

    print(df_categorical[df_categorical[var].isna()][var])
    df_categorical_imp.loc[df_categorical_imp[var].isna(), var] = mode

### 8.3 Combine imputed dataframes

In [ ]:
df_td_imputed = pd.concat([df_categorical_imp, df_numerical_imp], axis=1)
df_td_imputed
# df_categorical_imp

### 8.4 Convert categorical data to dummy variables (one-hot encoding)

In [ ]:
df_filtered

In [ ]:
df_td_imputed["intubated"] = df_filtered.reset_index(drop=True)["intubated"].astype(int)
df_td_imputed["died"] = df_filtered.reset_index(drop=True)["died"].astype(int)
df_td_imputed["hfno_failure"] = df_filtered.reset_index(drop=True)["hfno_failure"].astype(int)

df_td_imputed = pd.get_dummies(df_td_imputed, dtype=int)
died_col = df_td_imputed.pop('died')
df_td_imputed.insert(0, "died", died_col)
intubated_col = df_td_imputed.pop('intubated')
df_td_imputed.insert(0, "intubated", intubated_col)
hfno_failure_col = df_td_imputed.pop('hfno_failure')
df_td_imputed.insert(0, "hfno_failure", hfno_failure_col)
df_td_imputed

### 8.5 Add Relevant Timestamps

In [ ]:
df_td_imputed.insert(0, "iv_time", time_df['iv_time'].values)
df_td_imputed.insert(0, "deathtime", time_df['deathtime'].values)
df_td_imputed.insert(0, "final_starttime", time_df['final_starttime'].values)
df_td_imputed.insert(0, "final_endtime", time_df['final_endtime'].values)
df_td_imputed

### 8.6 Save imputed dataframe to CSV file

In [ ]:
df_td_imputed.to_csv(f"filtered_data/{dataset}_tabledata_imputed.csv")

#### 8.6.1 Save separate table containing Sepsis components to CSV file

In [ ]:
df_s3_imputed = df_td_imputed

df_s3_imputed.insert(0, "s3_antibiotic_time", sepsis_df["s3_antibiotic_time"].values)
df_s3_imputed.insert(0, "s3_sofa_time", sepsis_df["s3_sofa_time"].values)
df_s3_imputed.insert(0, "s3_sofa_score", sepsis_df["s3_sofa_score"].values)
df_s3_imputed.insert(0, "s3_respiration", sepsis_df["s3_respiration"].values)
df_s3_imputed.insert(0, "s3_coagulation", sepsis_df["s3_coagulation"].values)
df_s3_imputed.insert(0, "s3_liver", sepsis_df["s3_liver"].values)
df_s3_imputed.insert(0, "s3_cardiovascular", sepsis_df["s3_cardiovascular"].values)
df_s3_imputed.insert(0, "s3_cns", sepsis_df["s3_cns"].values)
df_s3_imputed.insert(0, "s3_renal", sepsis_df["s3_renal"].values)

df_s3_imputed.insert(0, "sofa_respiration", sepsis_df["sofa_respiration"].values)
df_s3_imputed.insert(0, "sofa_coagulation", sepsis_df["sofa_coagulation"].values)
df_s3_imputed.insert(0, "sofa_liver", sepsis_df["sofa_liver"].values)
df_s3_imputed.insert(0, "sofa_cardiovascular", sepsis_df["sofa_cardiovascular"].values)
df_s3_imputed.insert(0, "sofa_cns", sepsis_df["sofa_cns"].values)
df_s3_imputed.insert(0, "sofa_renal", sepsis_df["sofa_renal"].values)

# Only keep rows for patients with Sepsis-3:
# df_s3_imputed = df_s3_imputed[df_s3_imputed.sepsis3 == 1]
df_s3_imputed

In [ ]:
df_s3_imputed.to_csv(f"filtered_data/{dataset}_tabledata_imputed_sepsis3.csv")

## 9. Process post-HFNO data

Applies the same pipeline as sections 2-8 (sparsity filter, correlated-feature removal, fluid-balance clip, median imputation, GCS binning, one-hot encoding) to the post-HFNO windows [4 h, 12 h, 24 h].  Saves iltered_data/mimiciv_tabledata_post_hfno_imputed.csv.

In [ ]:
from sklearn.impute import SimpleImputer

POST_CORRELATED_BASE = [
    'sbp_mean',
    'dbp_mean',
    'hematocrit_last',
    'mcv_last',
    'rbc_last',
    'bun_last',
    'chloride_last',
    'baseexcess_last',
    'totalco2_last',
    'glucose_last',
    'bicarbonate_last'
]

gcs_bins_post   = [2, 8, 12, 14, 15]
gcs_labels_post = ["Severe (3-8)", "Moderate (9-12)", "Mild (13-14)", "Normal (15)"]

post_raw = pd.read_csv('filtered_data/mimiciv_tabledata_post_hfno_raw.csv')
post_raw = post_raw[post_raw['stay_id'].isin(time_df['stay_id'])].copy()
print(f"Post-HFNO raw (cohort-filtered): {post_raw.shape}")

id_to_fst = dict(zip(time_df['stay_id'], time_df['final_starttime']))
post_raw['final_starttime'] = post_raw['stay_id'].map(id_to_fst)

all_post = post_raw[['stay_id', 'final_starttime']].copy()

for h in [4, 12, 24]:
    s = f"post{h}h"
    print(f"\n--- {h}h window ---")

    post_cols = (
        dataset_features.get(f'vitals_mean_{s}', [])
        + dataset_features.get(f'last_{s}', [])
        + [f'gcs_min_{s}', f'max_flow_rate_{s}']
    )
    post_cols = [c for c in post_cols if c in post_raw.columns]
    tf = post_raw[['stay_id'] + post_cols].copy()

    corr_to_drop = [f'{b}_{s}' for b in POST_CORRELATED_BASE if f'{b}_{s}' in tf.columns]
    tf = tf.drop(columns=corr_to_drop)
    print(f"  Correlated features dropped: {len(corr_to_drop)}")

    feat_missing = tf.drop(columns=['stay_id']).isna().mean()
    sparse = feat_missing[feat_missing > 0.25].index.tolist()
    tf = tf.drop(columns=sparse)
    print(f"  Sparse features dropped (>25%): {len(sparse)}")

    fb_col = f'fluidbalance_{s}'
    if fb_col in tf.columns:
        tf[fb_col] = tf[fb_col].clip(lower=-4000, upper=5000)

    gcs_col     = f'gcs_min_{s}'
    gcs_bin_col = f'gcs_binned_{s}'
    if gcs_col in tf.columns:
        tf[gcs_bin_col] = pd.cut(tf[gcs_col], bins=gcs_bins_post, labels=gcs_labels_post)
        tf = tf.drop(columns=[gcs_col])

    cat_cols = [gcs_bin_col] if gcs_bin_col in tf.columns else []
    num_cols = [c for c in tf.columns if c != 'stay_id' and c not in cat_cols]

    imp = SimpleImputer(strategy='median')
    num_imp = imp.fit_transform(tf[num_cols])
    tf_imp = pd.DataFrame(num_imp, columns=num_cols)

    if cat_cols:
        for c in cat_cols:
            tf_imp[c] = tf[c].values
    tf_imp['stay_id'] = tf['stay_id'].values
    tf_imp = pd.get_dummies(tf_imp, columns=cat_cols, dtype=int)

    new_cols = [c for c in tf_imp.columns if c != 'stay_id']
    all_post = all_post.merge(tf_imp[['stay_id'] + new_cols], on='stay_id', how='left')
    print(f"  Columns added for {h}h: {len(new_cols)}")

print(f"\nCombined post-HFNO processed shape: {all_post.shape}")
all_post.to_csv('filtered_data/mimiciv_tabledata_post_hfno_imputed.csv', index=False)
print("Saved -> filtered_data/mimiciv_tabledata_post_hfno_imputed.csv")
all_post.head()